In [8]:
# Задание task_03_04_07.
#
# Выполнил: Шаповалова С.С
# Группа: ЦИБ-251


import csv


class NoSuchCountryError(Exception):
    def __init__(self, message):
        super().__init__(message)


class IllegalArgumentError(ValueError):
    pass


def load_data(filename):
    """Загрузить данные ВВП на душу населения из csv-файла 'filename'."""
    result = []

    # Открываем файл для чтения
    file = open(filename, mode="r", encoding="utf-8")
    reader = csv.reader(file)

    for row in reader:
        # Проверяем, что строка не пустая и в ней есть хотя бы 2 колонки
        if len(row) < 2:
            continue

        name = row[0].strip()
        gdp_str = row[1].strip()

        # Пропускаем, если значение ВВП пустое
        if gdp_str == "":
            continue

        # Пытаемся превратить строку в число
        try:
            gdp_val = float(gdp_str)
            # Создаем обычный словарь и добавляем в список
            country_dict = {"name": name, "gdp": gdp_val}
            result.append(country_dict)
        except ValueError:
            # Если там не число (например, текст "нет данных"), просто пропускаем строку
            continue

    file.close()
    return result


def search(data, criteria):
    """Выполнить поиск государства-значения в 'data' по критерию 'criteria'."""
    # Если список пустой, искать нечего
    if len(data) == 0:
        raise NoSuchCountryError("Данные отсутствуют.")

    # 1. Поиск максимального ВВП
    if criteria == "-max-":
        max_item = data[0]
        for item in data:
            if item["gdp"] > max_item["gdp"]:
                max_item = item
        return max_item

    # 2. Поиск минимального ВВП
    elif criteria == "-min-":
        min_item = data[0]
        for item in data:
            if item["gdp"] < min_item["gdp"]:
                min_item = item
        return min_item

    # 3. Поиск по названию страны
    else:
        for item in data:
            if item["name"] == criteria:
                return item

        # Если ни один if не сработал и страна не найдена
        raise NoSuchCountryError(
            "Значение параметра 'criteria' может быть "
            "одним из:\n"
            '- "-max-": государство с максимальным ВВП на душу населения;\n'
            '- "-min-": государство с мнимальным ВВП на душу населения;\n'
            '- "Russian Federation": название государства.')


def save_data(filename, data, criteria):
    """Сохранить данные 'data' в csv-файл 'filename' по критерию 'criteria'."""
    try:
        # Разбиваем строку по знаку равенства
        parts = criteria.split("=")
        method = parts[0]
        value = parts[1]

        filtered_data = []

        # Функция для сортировки, понятная новичку
        def get_gdp(item):
            return item["gdp"]

        # Определение операции
        if method == "top":
            x = int(value)
            if x <= 0:
                raise ValueError()
            # Сортируем по убыванию
            data.sort(key=get_gdp, reverse=True)
            # Берем первые X элементов
            filtered_data = data[:x]

        elif method == "tail":
            x = int(value)
            if x <= 0:
                raise ValueError()
            # Сортируем по возрастанию
            data.sort(key=get_gdp)
            # Берем первые X элементов
            filtered_data = data[:x]

        elif method == "greater":
            x = float(value)
            # Отбираем те, что больше X
            for item in data:
                if item["gdp"] > x:
                    filtered_data.append(item)
            # Сортируем результат по убыванию
            filtered_data.sort(key=get_gdp, reverse=True)

        elif method == "less":
            x = float(value)
            # Отбираем те, что меньше X
            for item in data:
                if item["gdp"] < x:
                    filtered_data.append(item)
            # Сортируем результат по возрастанию
            filtered_data.sort(key=get_gdp)

        else:
            raise ValueError()

        # Сохранение в файл
        file = open(filename, mode="w", newline="", encoding="utf-8")
        writer = csv.writer(file)
        for item in filtered_data:
            writer.writerow([item["name"], item["gdp"]])
        file.close()

    except Exception:
        raise IllegalArgumentError(
            "Значение параметра 'criteria' может быть "
            "одним из:\n"
            '- "top=X": первые X государств по ВВП на душу населения'
            ' (целое число > 0, по убыванию значения);\n'
            '- "tail=X": последние X государств по ВВП на душу населения'
            ' (целое число > 0, по возрастанию значения);\n'
            '- "greater=X": список государств с ВВП на душу населения, больше'
            ' чем X (вещ. число, по убыванию значения);\n'
            '- "less=X": список государств с ВВП на душу населения, меньше'
            ' чем X (вещ. число, по возрастанию значения).')


# Блок обработки исключений при запуске программы
try:
    filename = input("Введите имя файла для чтения: ")
    # filename = "gdp_per_capita_2016.csv"
    save_filename = input("Введите имя файла для сохранения: ")
    # save_filename = "output.csv"

    data = load_data(filename)

    print("\n--- Результаты поиска ---")
    try:
        print(search(data, criteria="-max-"))
    except NoSuchCountryError as e:
        print("Ошибка:", e)

    try:
        print(search(data, criteria="-min-"))
    except NoSuchCountryError as e:
        print("Ошибка:", e)

    try:
        print(search(data, criteria="Russian Federation"))
    except NoSuchCountryError as e:
        print("Ошибка:", e)

    print("\n--- Сохранение ---")
    save_data(save_filename, data, criteria="top=5")
    print("Данные успешно сохранены!")

except FileNotFoundError:
    print("Ошибка: Входной файл не найден. Проверьте имя файла.")
except IllegalArgumentError as e:
    print("Ошибка в критериях сохранения:\n", e)
except Exception as e:
    print("Произошла какая-то ошибка:", e)


Введите имя файла для чтения: gdp_per_capita_2016.csv
Введите имя файла для сохранения: output.csv

--- Результаты поиска ---
{'name': 'Luxembourg', 'gdp': 102831.321482811}
{'name': 'Burundi', 'gdp': 285.727442064713}
{'name': 'Russian Federation', 'gdp': 8748.36450405452}

--- Сохранение ---
Данные успешно сохранены!
